In [ ]:
import glob
import torch
import sys
import numpy as np
from torch.utils.data import Dataset, DataLoader
from scipy.io import wavfile
from scipy import signal
from sklearn.decomposition import PCA


sys.path.append('/om2/user/bjmedina/')

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params

class AudioTextureDataset(Dataset):
    def __init__(self, filepaths, sr=20000, rms_level=0.04, duration=2, device='cuda'):
        self.filepaths = filepaths
        self.sr = sr
        self.rms_level = rms_level
        self.duration = duration
        self.device = device

        self.flatten_stat_dict = FlattenStats(statistics_dict)
        coch_params, mod_params, octmod_params = model_params.update_param_dicts(duration)
        coch_params['rep_kwargs']['coch_filter_kwargs']['n'] = 30
        self.texture_model = TextureModel(coch_params, mod_params, octmod_params,
                                          statistics_dict=statistics_dict).to(device)

    def __len__(self):
        return len(self.filepaths)

    def rms(self, sound):
        return np.sqrt(np.mean(np.square(sound)))

    def process_sound(self, filepath):
        original_sr, sound = wavfile.read(filepath)
        if sound.ndim > 1:
            sound = sound.mean(axis=1)
        sound = signal.resample_poly(sound, self.sr, original_sr, axis=0)
        sound = sound - np.mean(sound)
        sound = self.rms_level * sound / self.rms(sound)
        sound = torch.from_numpy(sound).float().unsqueeze(0).to(self.device)
        return sound

    def __getitem__(self, idx):
        filepath = self.filepaths[idx]
        sound = self.process_sound(filepath)
        sound = sound.unsqueeze(0)  # shape [1, 1, audio_samples]
        stats_dict = self.texture_model(sound)
        stats = self.flatten_stat_dict(stats_dict).squeeze(0)
        return stats

    def compute_texture_statistics(self, batch_size=32):
        dataloader = DataLoader(self, batch_size=batch_size, shuffle=False)

        stats_list = []
        for batch_stats in dataloader:
            stats_list.extend(batch_stats.cpu().numpy())

        return stats_list

import torch
import torch.nn as nn

class DistanceMemoryModel(nn.Module):
    def __init__(self, encoding_model, noise_variance=1.0, criterion=0.5, device='cpu'):
        super(DistanceMemoryModel, self).__init__()
        
        self.encoding_model = encoding_model
        self.noise_variance = noise_variance
        self.criterion = criterion
        self.device = device
        self.memory_bank = []
        self.debug_mode = False

        self.probe_reps = []
        self.memory_snapshots = []
        self.decisions = []
        self.trial_indices = []
        self.pca_result = None
        self.pca_ready = False

        self.filenames_seen = []
        self.probe_filenames = []

    def _toggle_debug(self):

        self.debug_mode = not self.debug_mode

        print(f"Debug flag is set to {self.debug_mode}")

    def clear_memory(self):
        """Clear all stored memory representations."""
        self.memory_bank = []
        self.memory_snapshots = []
        self.probe_reps = []
        self.decisions = []
        self.trial_indices = []
        self.filenames_seen = []
        self.probe_filenames = []


    def encode_sound(self, sound):
        """Encode a single sound into its representation (filepath expected)."""
        with torch.no_grad():
            rep = self.encoding_model(sound).squeeze(0)
        return rep

    def apply_noise_to_memory(self):
        """Simulate memory drift by applying noise to memory representations."""
        noisy_bank = []
        for rep in self.memory_bank:
            noise = torch.randn_like(rep) * self.noise_variance
            noisy_rep = rep + noise
            noisy_bank.append(noisy_rep)
        self.memory_bank = noisy_bank

    def forward(self, sound):
        """
        Process a single sound, decide recognition, store it in memory.
        
        Args:
            sound: raw audio tensor.
        
        Returns:
            decision: tensor([1]) if recognized, tensor([0]) otherwise.
        """
        current_rep = self.encode_sound(sound)

        if self.debug_mode:
            print(f"Current representation of {sound}: {current_rep[:2]}")

        # First presentation, no recognition possible
        if not self.memory_bank:
            decision = torch.tensor([0], device=self.device)
        else:
            # Compute distances
            memory_tensor = torch.stack(self.memory_bank)
            dists = torch.cdist(current_rep.unsqueeze(0), memory_tensor, p=2)

            if self.debug_mode:
                print(f"All distances are {dists}")
                
            min_dist = dists.min()

            if self.debug_mode:
                print(f"The minimum distance is {min_dist}\n")
                
            # Decision based on criterion
            decision = (min_dist <= self.criterion).float().unsqueeze(0)

        # Apply noise (drift) to existing memories
        self.apply_noise_to_memory()

        # Store current representation into memory
        self.memory_bank.append(current_rep)

        # Save internal state for visualization
        self.probe_reps.append(current_rep.detach().cpu())
        self.memory_snapshots.append(torch.stack(self.memory_bank).detach().cpu())
        self.decisions.append(decision.item())
        self.trial_indices.append(len(self.decisions) - 1)
        self.probe_filenames.append(sound)
        
        # Store ground-truth filename
        self.filenames_seen.append(sound)

        def compute_pca_projection(self):
            from sklearn.decomposition import PCA
            all_reps = torch.vstack(self.memory_snapshots + self.probe_reps)
            pca = PCA(n_components=2)
            self.pca_result = pca.fit_transform(all_reps.numpy())
            self.pca_ready = True
        
            # Compute indices for later slicing
            self._mem_lens = [m.shape[0] for m in self.memory_snapshots]
            self._mem_offsets = [sum(self._mem_lens[:i]) for i in range(len(self._mem_lens))]
            self._probe_indices = [
                offset + self._mem_lens[i] for i, offset in enumerate(self._mem_offsets)
            ]

        return decision

    def animate_trials(self, save_path=None):
        import matplotlib.pyplot as plt
        import matplotlib.animation as animation
        import numpy as np
    
        fig, ax = plt.subplots(figsize=(6, 6))
        sc_mem = ax.scatter([], [], c='blue', label='Memory (noisy)', alpha=0.6)
        sc_probe = ax.scatter([], [], c='red', marker='X', s=100, label='Probe')
        text = ax.text(0.05, 0.95, '', transform=ax.transAxes, fontsize=12, va='top')
    
        # Determine axis limits using all memory + probe representations
        all_points = torch.cat([torch.cat(self.memory_snapshots), torch.stack(self.probe_reps)], dim=0)[:, :2]
        ax.set_xlim(all_points[:, 0].min() - 1, all_points[:, 0].max() + 1)
        ax.set_ylim(all_points[:, 1].min() - 1, all_points[:, 1].max() + 1)
        ax.set_xlabel("Feature 1")
        ax.set_ylabel("Feature 2")
        ax.set_title("Recognition Decisions Over Time")
        ax.legend(loc='lower right')
        ax.grid(True)
    
        # Storage for fading trail scatter collections
        memory_trails = []
    
        def update(frame):
            # Clear previous trail collections from plot
            for coll in memory_trails:
                coll.remove()
            memory_trails.clear()
    
            # --- Draw fading memory trail ---
            num_past = frame
            if num_past > 0:
                alphas = np.linspace(0.05, 0.4, num_past)  # older = more transparent
                for i in range(num_past):
                    mem_2d = self.memory_snapshots[i][:, :2].numpy()
                    trail = ax.scatter(mem_2d[:, 0], mem_2d[:, 1], c='gray', alpha=alphas[i], s=15, label='_nolegend_')
                    memory_trails.append(trail)
    
            # --- Current memory and probe ---
            mem_2d = self.memory_snapshots[frame][:, :2].numpy()
            probe_2d = self.probe_reps[frame][:2].numpy()
    
            sc_mem.set_offsets(mem_2d)
            sc_probe.set_offsets(probe_2d.reshape(1, -1))
    
            # --- Trial info ---
            filename = self.probe_filenames[frame]
            model_said = self.decisions[frame]
            ground_truth = filename in self.filenames_seen[:frame]
            correctness = 'correct' if model_said == ground_truth else 'incorrect'
    
            text.set_text(
                f"Trial {frame+1}: "
                f"{'YES' if model_said else 'NO'} (model) | "
                f"{'YES' if ground_truth else 'NO'} (truth) {correctness}\n"
                f"{filename.split('/')[-1]}"
            )
    
            return [sc_mem, sc_probe, text] + memory_trails
    
        ani = animation.FuncAnimation(
            fig, update, frames=len(self.trial_indices), interval=1000, blit=True
        )
    
        if save_path:
            ani.save(save_path, dpi=150, fps=1.0)
            print(f"Animation saved to {save_path}")
        else:
            from IPython.display import HTML
            return HTML(ani.to_jshtml())

In [ ]:
import torch
import torch.nn as nn
from scipy.io import wavfile
from scipy import signal
import numpy as np
import glob

class AudioTextureEncoder(nn.Module):
    def __init__(self, statistics_dict, model_params, sr=20000, rms_level=0.04, duration=2.0, device='cuda'):
        super().__init__()
        self.sr = sr
        self.rms_level = rms_level
        self.duration = duration
        self.device = device

        self.flatten_stat_dict = FlattenStats(statistics_dict)
        coch_params, mod_params, octmod_params = model_params.update_param_dicts(duration)
        coch_params['rep_kwargs']['coch_filter_kwargs']['n'] = 30
        self.texture_model = TextureModel(coch_params, mod_params, octmod_params,
                                          statistics_dict=statistics_dict).to(device)

    def forward(self, filepath):
        """
        Accepts a string path to a .wav file, trims or drops based on length,
        returns a flattened texture representation tensor.
        """
        try:
            target_len = int(self.sr * self.duration)  # expected length in samples
            original_sr, sound = wavfile.read(filepath)
    
            if sound.ndim > 1:
                sound = sound.mean(axis=1)
    
            # Resample
            sound = signal.resample_poly(sound, self.sr, original_sr, axis=0)
    
            # Check length
            if len(sound) < target_len:
                # Drop short sounds
                raise ValueError(f"Sound too short after resampling: {len(sound)} < {target_len}")
            elif len(sound) > target_len:
                # Random crop
                start = np.random.randint(0, len(sound) - target_len + 1)
                sound = sound[start:start + target_len]
    
            # Normalize to zero mean and target RMS
            sound = sound - np.mean(sound)
            rms = np.sqrt(np.mean(np.square(sound)))
            sound = self.rms_level * sound / (rms + 1e-6)
    
            # Prepare tensor
            sound_tensor = torch.from_numpy(sound).float().unsqueeze(0).unsqueeze(0).to(self.device)
    
            with torch.no_grad():
                stats_dict = self.texture_model(sound_tensor)
                stats = self.flatten_stat_dict(stats_dict).squeeze(0)
            return stats
    
        except Exception as e:
            print(f"Skipping {filepath}: {e}")
            return None

class PCASpace(nn.Module):
    def __init__(self, encoder, n_components=2, device='cpu'):
        super().__init__()
        self.encoder = encoder
        self.n_components = n_components
        self.device = device
        self.pca = None

    def fit(self, filepaths):
        """Fit PCA on a list of audio filepaths."""
        reps = []
        for path in filepaths:
            rep = self.encoder(path).detach().cpu().numpy()
            reps.append(rep)
        X = np.vstack(reps)
        self.pca = PCA(n_components=self.n_components)
        self.pca.fit(X)

    def transform(self, filepaths):
        """Project new filepaths into PCA space."""
        assert self.pca is not None, "You must call .fit() first"
        reps = []
        for path in filepaths:
            rep = self.encoder(path).detach().cpu().numpy()
            reps.append(rep)
        return self.pca.transform(np.vstack(reps))

    def fit_transform(self, filepaths):
        """Convenience method to fit and then project."""
        self.fit(filepaths)
        return self.transform(filepaths)

    def forward(self, filepath):
        """Encode + project a single file."""
        assert self.pca is not None, "Call fit() first."
        rep = self.encoder(filepath).detach().cpu().numpy().reshape(1, -1)
        proj = self.pca.transform(rep)
        return torch.tensor(proj, dtype=torch.float32).squeeze(0).to(self.device)

class ZScoreSpace(nn.Module):
    def __init__(self, encoder, device='cpu', eps=1e-6):
        super().__init__()
        self.encoder = encoder
        self.device = device
        self.eps = eps
        self.mean = None
        self.std = None

    def fit(self, filepaths):
        """
        Fit the z-scoring parameters (mean and std) using a list of audio filepaths.
        """
        reps = []
        for path in filepaths:
            rep = self.encoder(path)
            if rep is not None:
                reps.append(rep.detach().cpu().numpy())

        X = np.vstack(reps)
        self.mean = X.mean(axis=0)
        self.std = X.std(axis=0)

    def transform(self, filepaths):
        """
        Z-score transform a list of filepaths.
        """
        assert self.mean is not None and self.std is not None, "You must call .fit() first"
        reps = []
        for path in filepaths:
            rep = self.encoder(path)
            if rep is not None:
                rep = rep.detach().cpu().numpy()
                z = (rep - self.mean) / (self.std + self.eps)
                reps.append(z)
        return np.vstack(reps)

    def forward(self, filepath):
        """
        Encode and z-score a single file.
        """
        assert self.mean is not None and self.std is not None, "Call fit() first."
        rep = self.encoder(filepath)
        if rep is None:
            return None
        rep = rep.detach().cpu().numpy()
        z = (rep - self.mean) / (self.std + self.eps)
        return torch.tensor(z, dtype=torch.float32).to(self.device)

sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")

In [ ]:
print(len(ALL_SOUNDS))
dataset = AudioTextureDataset(texture_list)
stats_list = dataset.compute_texture_statistics()
print(len(stats_list), stats_list[0].shape)

In [ ]:
texture_model = AudioTextureEncoder(
    statistics_dict=statistics_dict,
    model_params=model_params,
    sr=20000,
    rms_level=0.04,
    duration=2.0,
    device='cuda'
)

ex = texture_model(sounds_list[0])

In [ ]:
projector = PCASpace(texture_model, n_components=None)  # or use encoder output dim explicitly
projector.fit(texture_list)

dim = (projector.pca.explained_variance_ratio_.cumsum() > 0.9).tolist().index(True)

In [ ]:
projector = PCASpace(texture_model, n_components=dim)

# Fit on example_trials
projector.fit(texture_list)

# Project those same trials or new ones
coords = projector.transform(texture_list)

In [ ]:
zscore_projector = ZScoreSpace(texture_model)
zscore_projector.fit(texture_list)

In [ ]:
memory_model = DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=0.1,
    criterion=10,
    device='cuda'
)

memory_model.clear_memory()

example_trials = [sounds_list[0],
                  sounds_list[1],
                  sounds_list[2],
                  sounds_list[3],
                  sounds_list[0],
                  sounds_list[1]]

for j, path in enumerate(example_trials):
    decision = memory_model(path)
    true_answer = 'YES' if example_trials[j] in example_trials[:j] else 'NO'
    print(f"{path.split('/')[-1]} ==> Model {'YES' if decision.item() else 'NO'}, True {true_answer}")

In [ ]:
memory_model.animate_trials()

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

probe_reps = []
memory_snapshots = []
decisions = []

memory_model.clear_memory()

example_trials = [sounds_list[0],
                  sounds_list[1],
                  sounds_list[2],
                  sounds_list[3],
                  sounds_list[0],
                  sounds_list[1],
                  sounds_list[4],
                  sounds_list[5],
                  sounds_list[6],
                  sounds_list[7],
                  sounds_list[3],
                 ]

for j, path in enumerate(example_trials):
    current_rep = memory_model.encode_sound(path).detach().cpu()
    memory_snapshot = torch.stack(memory_model.memory_bank).detach().cpu() if memory_model.memory_bank else torch.empty(0, current_rep.shape[0])
    
    decision = memory_model(path)
    true_answer = 'YES' if example_trials[j] in example_trials[:j] else 'NO'


    decisions.append(decision.item())


In [ ]:
memory_model.animate_trials()

In [ ]:
def plot_trial_pca(trial_idx):
    start_idx = mem_offsets[trial_idx]
    end_idx = start_idx + mem_lens[trial_idx]
    
    memory_2d = pca_result[start_idx:end_idx]
    probe_2d = pca_result[probe_indices[trial_idx]]

    plt.figure(figsize=(6, 6))
    plt.scatter(memory_2d[:, 0], memory_2d[:, 1], c='blue', label='Memory (noisy)')
    plt.scatter(probe_2d[0], probe_2d[1], c='red', marker='X', s=100, label='Probe')

    plt.title(f"Trial {trial_idx+1} – Decision: {'YES' if decisions[trial_idx] else 'NO'} - GC: ")
    plt.legend()
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True)
    plt.axis('equal')
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots(figsize=(6, 6))

sc_mem = ax.scatter([], [], c='blue', label='Memory (noisy)', alpha=0.6)
sc_probe = ax.scatter([], [], c='red', marker='X', s=100, label='Probe')

text = ax.text(0.05, 0.95, '', transform=ax.transAxes, fontsize=12, va='top')
ax.set_xlim(pca_result[:, 0].min() - 1, pca_result[:, 0].max() + 1)
ax.set_ylim(pca_result[:, 1].min() - 1, pca_result[:, 1].max() + 1)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Memory Drift and Recognition Decisions Over Time")
ax.legend(loc='lower right')
ax.grid(True)

def update(frame):
    start_idx = mem_offsets[frame]
    end_idx = start_idx + mem_lens[frame]
    memory_2d = pca_result[start_idx:end_idx]
    probe_2d = pca_result[probe_indices[frame]]

    sc_mem.set_offsets(memory_2d)
    sc_probe.set_offsets(probe_2d)

    text.set_text(f"Trial {frame+1} – Decision: {'YES' if decisions[frame] else 'NO'}")

    return sc_mem, sc_probe, text

ani = animation.FuncAnimation(
    fig, update, frames=len(probe_reps), interval=800, blit=True
)

In [ ]:
from IPython.display import HTML
HTML(ani.to_jshtml())